# DeepWhale End-to-End Pipeline

This notebook runs the full sequence:
1) Collect raw whale tx data
2) Build address features
3) Label addresses
4) Cluster profiles
5) Train classifier
6) Detect anomalies
7) Quick sanity previews

Prereqs: `.env` with `ALCHEMY_URL`, and `pip install -r requirements.txt` in this env. Run cells top-to-bottom.

In [3]:
# Setup paths and helpers
import os, json, subprocess, sys, textwrap
from pathlib import Path
import pandas as pd

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()  # fallback when __file__ is unavailable (e.g., in notebooks)
DATA_DIR = BASE_DIR / "data_new"
MODELS_DIR = BASE_DIR / "models"
DATA_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

def run(cmd):
    print("$", " ".join(cmd))
    res = subprocess.run(cmd, cwd=BASE_DIR, text=True, capture_output=True)
    if res.stdout:
        print(res.stdout.strip())
    if res.stderr:
        print(res.stderr.strip())
    if res.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")

print("Base dir:", BASE_DIR)

Base dir: c:\D\UCU\ML\project


## 1) Collect raw whale transactions
Runs data_collection.py (uses ALCHEMY_URL). Adjust blocks inside the script if needed.

In [ ]:
try:
    run([sys.executable, "data_collection.py"])

except Exception as e:
    print("Skip/inspect: data_collection.py failed", e)

## 2) Feature engineering
Aggregates raw transactions to per-address features.

In [8]:
try:
    run([sys.executable, "feature_engineering.py"])

except Exception as e:
    print("Skip/inspect: feature_engineering.py failed", e)

$ c:\D\UCU\ML\project\.venv\Scripts\python.exe feature_engineering.py
Loading c:\D\UCU\ML\project\data_new\raw_whale_transactions.csv ...
  6426 raw transactions, 3288 unique senders
  Loaded 0 known exchange addresses
Building address features ...
  3274 addresses with features
Saved to c:\D\UCU\ML\project\data_new\address_features.csv
       tx_count_out  total_eth_out  total_usd_out    avg_tx_eth  median_tx_eth    max_tx_eth   std_tx_eth  unique_receivers  exchange_ratio  top1_receiver_ratio  round_number_ratio  avg_gas_gwei  gas_variability  hour_entropy  active_days  avg_interval_hours  net_flow_eth     block_span  is_known_exchange
count   3274.000000    3274.000000   3.274000e+03   3274.000000    3274.000000   3274.000000  3274.000000       3274.000000          3274.0          3274.000000         3274.000000    3274.00000      3274.000000   3274.000000  3274.000000          490.000000   3274.000000    3274.000000             3274.0
mean       1.952963     230.151773   5.226298e+

## 3) Labeling
Rule-based labels for each address.

In [9]:
try:
    run([sys.executable, "labeling.py"])

except Exception as e:
    print("Skip/inspect: labeling.py failed", e)

$ c:\D\UCU\ML\project\.venv\Scripts\python.exe labeling.py
Loading features from c:\D\UCU\ML\project\data_new\address_features.csv ...
  3274 addresses

Label distribution:
  unknown_whale          3017 ( 92.2%)  avg_confidence=0.30
  accumulator             213 (  6.5%)  avg_confidence=0.74
  active_trader            44 (  1.3%)  avg_confidence=0.79

Saved labeled data to c:\D\UCU\ML\project\data_new\labeled_addresses.csv

--- active_trader (sample) ---
                                   address  tx_count_out  exchange_ratio  active_days  net_flow_eth  unique_receivers         label  label_confidence
0x0003B5Aa5E30E97FcC596BB5D0F3A75255E08D4e            17             0.0           10    12471.8346                10 active_trader            0.6875
0x0D0707963952f2fBA59dD06f2b425ace40b492Fe            21             0.0           18      406.2447                15 active_trader            0.9375

--- unknown_whale (sample) ---
                                   address  tx_count_out  e

## 4) Clustering
Unsupervised profiling (KMeans + DBSCAN) and plots.

In [10]:
try:
    run([sys.executable, "clustering.py"])

except Exception as e:
    print("Skip/inspect: clustering.py failed", e)

$ c:\D\UCU\ML\project\.venv\Scripts\python.exe clustering.py
Loaded 3274 addresses from labeled_addresses.csv

[KMeans] Finding best k ...
  k=2  silhouette=0.511
  k=3  silhouette=0.524
  k=4  silhouette=0.476
  k=5  silhouette=0.352
  k=6  silhouette=0.364
  k=7  silhouette=0.370
  k=8  silhouette=0.308
Best silhouette at k=3 (0.524), but k=4 (0.476) is within 9.2% â€” choosing k=4 for richer segmentation

[PCA] Plotting KMeans clusters ...
  Saved cluster_pca_kmeans.png
[t-SNE] Plotting KMeans clusters (may take a moment) ...
  Saved cluster_tsne_kmeans.png
[Radar] Plotting cluster behaviour profiles ...
  Saved cluster_radar.png

=== Cluster Profiles (medians) ===
                tx_count_out  total_eth_out  avg_tx_eth  unique_receivers  exchange_ratio  round_number_ratio  active_days  net_flow_eth
kmeans_cluster                                                                                                                          
0                        1.5         58.860      

## 5) Classification
Train whale classifier (XGBoost + RandomForest) and save bundle.

In [6]:
try:
    run([sys.executable, "classification.py"])

except Exception as e:
    print("Skip/inspect: classification.py failed", e)

$ c:\D\UCU\ML\project\.venv\Scripts\python.exe classification.py
Loaded 6643 labeled addresses
  Kept 6638 (dropped 5 low-confidence)
  Train: 5310  |  Test: 1328
Classes: ['accumulator', 'active_trader', 'exchange_depositor', 'unknown_whale']

[XGBoost] Training ...
  F1 macro (XGBoost): 1.0000
                    precision    recall  f1-score   support

       accumulator       1.00      1.00      1.00       531
     active_trader       1.00      1.00      1.00         1
exchange_depositor       1.00      1.00      1.00        85
     unknown_whale       1.00      1.00      1.00       711

          accuracy                           1.00      1328
         macro avg       1.00      1.00      1.00      1328
      weighted avg       1.00      1.00      1.00      1328

  Saved cm_xgboost.png
  Saved feature_importance.png
  Computing SHAP values ...
  Saved shap_summary.png

[Random Forest] Training ...
  F1 macro (Random Forest): 0.7498
                    precision    recall  f1-scor

## 6) Anomaly detection
Isolation Forest on address features + timeline plot (needs raw + features).

In [7]:
try:
    run([sys.executable, "anomaly_detection.py"])

except Exception as e:
    print("Skip/inspect: anomaly_detection.py failed", e)

$ c:\D\UCU\ML\project\.venv\Scripts\python.exe anomaly_detection.py
Loading address features ...
  6643 addresses

[Isolation Forest] Detecting anomalous whale addresses ...
  Anomalous addresses: 665 / 6643 (10.0%)
  Saved anomaly_scores_plot.png

=== Top 10 Most Anomalous Whale Addresses ===
                                   address  anomaly_score  tx_count_out  total_eth_out  exchange_ratio  unique_receivers
0x28C6c06298d514Db089934071355E5743bf21d60      -0.808602           270    396221.8877          0.5520               131
0xf30ba13e4b04Ce5dC4D254Ae5FA95477800F0EB0      -0.802878           303    118317.1494          0.0000               147
0xA9Ac43f5b5e38155A288d1A01D2cbc4478E14573      -0.800478           287    110670.4522          0.0000               109
0x3606f0F14828cBF4962A284a4bFF93Bc94b63665      -0.795222             7     56854.7255          0.0000                 2
0xE69f81b825d7Dc31ee9becef4DbEab5cf30e3Abb      -0.783559             6     47140.0000          0.89

## 7) Quick sanity previews
Show tail/head of key outputs if they exist.

In [ ]:
outputs = {
    "raw": DATA_DIR / "raw_whale_transactions.csv",
    "features": DATA_DIR / "address_features.csv",
    "labeled": DATA_DIR / "labeled_addresses.csv",
    "clustered": DATA_DIR / "clustered_addresses.csv",
    "anomaly": DATA_DIR / "anomaly_scores.csv",
    "model": MODELS_DIR / "whale_classifier.pkl",
}

for name, path in outputs.items():
    if path.exists():
        print(f"{name}: {path} ({path.stat().st_size/1024:.1f} KB)")
        if path.suffix == ".csv":
            df = pd.read_csv(path)
            print(df.head().to_string())
            print("...")
    else:
        print(f"{name}: missing — expected at {path}")